# SleepGuard: pre-cutoff features and LOPO evaluation

This rerun uses a **22:00 local recording-time prediction cutoff**. Glucose and ECG HRV are rebuilt from **20:00–22:00**; daytime low uses **10:00–22:00**. The supplied glucose means matched 22:00–00:00 for every original row, so the old predictors could not establish prediction at 22:00. The old HRV columns have no verified aggregation window and are replaced entirely.

**Remaining limitation:** the original 99-night selection, `Model_Eligible` flags, and `hypo` outcome labels are retained. The missing upstream label/merge notebook still needs an audit. This is an exploratory evaluation on that supplied cohort, not proof of leakage-free deployment performance.

Restart the kernel and run all cells. Each run writes a new folder under `results/lopo/`; historical outputs stay intact. Core implementation and temporal-regression tests live in `scripts/lopo_evaluation.py` and `tests/`. Raw ECG extraction lives in `scripts/build_presleep_hrv.py`.

In [1]:
from pathlib import Path
import importlib
import sys

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / 'scripts/lopo_evaluation.py').is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from scripts import lopo_evaluation
importlib.reload(lopo_evaluation)

CONFIG = lopo_evaluation.Config(n_bootstraps=1000, n_permutations=199)
print('Repository:', REPO_ROOT)
print(CONFIG)

Repository: /Users/aryan/Documents/GitHub/SleepGuard
Config(prediction_hour=22, daytime_start_hour=10, presleep_glucose_hours=2, low_mg_dl=70.0, cgm_interval_minutes=5, min_daytime_coverage=0.8, min_hrv_history=2, n_bootstraps=1000, bootstrap_seed=20261026, n_permutations=199, permutation_seed=20261102)


## Features and timing

- **A:** corrected glucose mean/slope, exact previous-calendar-night low (missing if that night is absent), daytime low, and mean low rate across the patient's earlier known labels in the supplied table.
- **B1 / B2 / B3:** A plus RMSSD z-score / LF-HF z-score / both. Each z-score uses that patient's **strictly earlier** rebuilt HRV values, with at least two earlier measurements and positive sample SD. First-night/insufficient histories are missing.
- A held-out patient's earlier measurements and outcomes may become available sequentially. They never enter classifier, scaler, or imputer fitting. This is LOPO with accumulating personal history, not history-free first-visit prediction.
- CGM features require 80% occupied five-minute bins. Missing daytime coverage stays missing. The daytime feature means any EGV below 70 mg/dL, not a duration-defined episode.
- HRV requires six accepted, non-overlapping five-minute ECG windows (30 minutes), with HRConfidence=100 and ECGNoise<0.001. Windows with more than 5% flagged RR intervals are rejected; at most 5% are interpolated within the same pre-cutoff window. Aggregate with the median. Source timestamps are saved and checked against 22:00 before evaluation.
- Every feature variant uses the same observed glucose/HRV cohort. Expected history missingness is imputed inside each fold using training patients only, with missingness indicators. Logistic and shallow-forest settings are fixed, not tuned on these results.

Rebuild inputs when needed with the commands in `README.md`. The evaluator requires the timestamp-audited rebuilt HRV file and never falls back to old HRV values.

In [2]:
out_dir, outputs = lopo_evaluation.run_evaluation(REPO_ROOT, CONFIG)
print('Completed run:', out_dir)

Using rebuilt pre-22:00 glucose/HRV. Stored outcome/cohort provenance remains exploratory.


Cohort: 62 nights, 10 patients, 22 low nights


LOPO fits complete; computing paired patient bootstrap


Permutation global: 1/199


Permutation global: 25/199


Permutation global: 50/199


... (10 outputs left out)

Permutation within_patient: 125/199


Permutation within_patient: 150/199


Permutation within_patient: 175/199


Permutation within_patient: 199/199
Saved results to: /Users/aryan/Documents/GitHub/SleepGuard/results/lopo/run_20261103T164918_024116Z
Completed run: /Users/aryan/Documents/GitHub/SleepGuard/results/lopo/run_20261103T164918_024116Z


## Pooled metrics

PR-AUC is **non-interpolated average precision (AP)**, not trapezoidal PR area. Prevalence is reported for context. AUROC and AP intervals use 1,000 paired patient-cluster bootstrap draws of fixed held-out predictions, preserving all nights and multiplicity for repeated patients. These intervals do not capture full refitting uncertainty; only ten patients were present in the original cohort.

In [3]:
display(outputs['pooled_metrics_ci.csv'].round(4))

,algorithm,feature_set,n_patients,n_nights,n_low_nights,prevalence,valid_bootstraps,skipped_single_class,pooled_auroc,auroc_ci_lower,auroc_ci_upper,pooled_pr_auc,pr_auc_ci_lower,pr_auc_ci_upper
0,logistic,A,10,62,22,0.3548,1000,0,0.4955,0.3367,0.6423,0.3366,0.2276,0.5219
1,logistic,B1,10,62,22,0.3548,1000,0,0.4739,0.2920,0.6400,0.3253,0.2164,0.5207
2,logistic,B2,10,62,22,0.3548,1000,0,0.4784,0.3144,0.6408,0.3292,0.2204,0.5251
3,logistic,B3,10,62,22,0.3548,1000,0,0.4568,0.2811,0.6160,0.3191,0.2121,0.5101
4,random_forest,A,10,62,22,0.3548,1000,0,0.4580,0.3483,0.6042,0.3340,0.1929,0.5472
5,random_forest,B1,10,62,22,0.3548,1000,0,0.4432,0.3275,0.5924,0.3206,0.1841,0.5267
6,random_forest,B2,10,62,22,0.3548,1000,0,0.4500,0.3336,0.5848,0.3285,0.1920,0.5362
7,random_forest,B3,10,62,22,0.3548,1000,0,0.4159,0.3045,0.5633,0.3175,0.1833,0.5110


## Paired gains over glucose baseline A

Primary comparison: **logistic B3−A**. All variants use identical patient resamples and identical evaluation rows. A positive point estimate alone does not establish improvement if its paired interval includes zero. Other feature variants and the random forest are exploratory.

In [4]:
display(outputs['paired_auc_differences.csv'].round(4))

,algorithm,comparison,valid_bootstraps,delta_auroc,auroc_ci_lower,auroc_ci_upper,delta_pr_auc,pr_auc_ci_lower,pr_auc_ci_upper
0,logistic,B1-A,1000,-0.0216,-0.0971,0.0234,-0.0113,-0.0440,0.0169
1,logistic,B2-A,1000,-0.0170,-0.0525,0.0146,-0.0074,-0.0239,0.0186
2,logistic,B3-A,1000,-0.0386,-0.1116,0.0070,-0.0175,-0.0525,0.0095
3,random_forest,B1-A,1000,-0.0148,-0.0638,0.0194,-0.0134,-0.0544,0.0079
4,random_forest,B2-A,1000,-0.0080,-0.0432,0.0199,-0.0055,-0.0397,0.0167
5,random_forest,B3-A,1000,-0.0420,-0.0941,-0.0008,-0.0165,-0.0790,0.0175


## Per-patient AUROC, PR-AUC and ΔAUROC

Single-class patients receive NA for both ranking metrics and remain in pooled results. Macro averages give equal weight to evaluable patients and report their denominator. No arbitrary 0.5-threshold counts table is produced.

In [5]:
display(outputs['patient_metrics.csv'].round(4))
display(outputs['patient_macro_metrics.csv'].round(4))

,subject,algorithm,feature_set,n_nights,n_low_nights,prevalence,status,auroc,pr_auc,delta_auroc_vs_A,delta_pr_auc_vs_A
0,c1s01,logistic,A,4,2,0.5,ok,0.00,0.4167,0.0,0.0
1,c1s01,logistic,B1,4,2,0.5,ok,0.00,0.4167,0.0,0.0
2,c1s01,logistic,B2,4,2,0.5,ok,0.00,0.4167,0.0,0.0
3,c1s01,logistic,B3,4,2,0.5,ok,0.00,0.4167,0.0,0.0
4,c1s01,random_forest,A,4,2,0.5,ok,0.25,0.5000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
75,c2s05,logistic,B3,3,3,1.0,undefined_single_class,NaN,NaN,NaN,NaN
76,c2s05,random_forest,A,3,3,1.0,undefined_single_class,NaN,NaN,NaN,NaN
77,c2s05,random_forest,B1,3,3,1.0,undefined_single_class,NaN,NaN,NaN,NaN
78,c2s05,random_forest,B2,3,3,1.0,undefined_single_class,NaN,NaN,NaN,NaN


,algorithm,feature_set,mean_patient_auroc,mean_patient_pr_auc,mean_patient_delta_auroc,evaluable_patients,total_patients
0,logistic,A,0.4292,0.5031,0.0000,8,10
1,logistic,B1,0.4132,0.4929,-0.0160,8,10
2,logistic,B2,0.4042,0.4985,-0.0250,8,10
3,logistic,B3,0.3507,0.4762,-0.0785,8,10
4,random_forest,A,0.4549,0.5098,0.0000,8,10
5,random_forest,B1,0.4181,0.4813,-0.0368,8,10
6,random_forest,B2,0.4368,0.4878,-0.0181,8,10
7,random_forest,B3,0.3806,0.4697,-0.0743,8,10


## Shuffled-label diagnostics

Refit primary logistic A and B3 for **199 global** and **199 within-patient** permutations. Rebuild prior-night low and earlier-night low rate from shuffled labels each time; keep the pre-cutoff physiological measurements fixed. Upper-tail Monte Carlo p-values use `(1 + exceedances) / (1 + permutations)`.

Global shuffling is an IID negative control. Within-patient shuffling preserves each patient's low count and therefore may retain patient-burden information in pooled scores. Neither null preserves serial dependence. Pooled LOPO null AUROC need not center exactly on 0.5; report the observed null honestly, together with per-patient macro AUROC. Chance-like performance cannot prove the absence of upstream leakage, and these diagnostics do not replace timestamp and label audits.

In [6]:
display(outputs['permutation_summary.csv'].round(4))

,shuffle,algorithm,metric,observed,null_mean,null_sd,null_2_5_percentile,null_97_5_percentile,p_greater_equal,n_permutations
0,global,logistic,pooled_auroc_A,0.4955,0.4538,0.1051,0.2575,0.6457,0.315,199
1,global,logistic,pooled_auroc_B3,0.4568,0.4645,0.1125,0.2200,0.6811,0.560,199
2,global,logistic,pooled_delta_auroc,-0.0386,0.0107,0.0672,-0.0908,0.1731,0.780,199
3,global,logistic,macro_auroc_A,0.4292,0.5196,0.1198,0.3036,0.7649,0.775,199
4,global,logistic,macro_auroc_B3,0.3507,0.5168,0.1233,0.2548,0.7741,0.910,199
5,global,logistic,macro_delta_auroc,-0.0785,-0.0028,0.0870,-0.1406,0.2042,0.820,199
6,within_patient,logistic,pooled_auroc_A,0.4955,0.5012,0.1112,0.2931,0.6962,0.550,199
7,within_patient,logistic,pooled_auroc_B3,0.4568,0.4926,0.1096,0.3040,0.6896,0.625,199
8,within_patient,logistic,pooled_delta_auroc,-0.0386,-0.0085,0.0534,-0.0907,0.1199,0.690,199
9,within_patient,logistic,macro_auroc_A,0.4292,0.4571,0.1499,0.1766,0.7582,0.600,199


## Cohort and source audits

These are cohort/coverage counts, not prediction counts at a threshold. Excluded nights stay in chronological history if their historical outcome is known; a missing HRV predictor does not erase a previously observed outcome.

In [7]:
display(outputs['cohort_counts.csv'])
display(outputs['excluded_nights.csv'])
print('Glucose values matching the old post-cutoff window:',
      outputs['glucose_timing_audit.csv']['stored_mean_matches_postcutoff'].sum())
print('All deliverables:', out_dir)
print('Read:', out_dir / 'REPORT.md')

,subject,input_nights,included_nights,low_nights
0,c1s01,7,4,2
1,c1s02,11,9,0
2,c1s03,10,7,1
3,c1s04,6,5,1
4,c1s05,13,9,4
5,c2s01,13,11,5
6,c2s02,9,3,2
7,c2s03,12,5,1
8,c2s04,11,6,3
9,c2s05,7,3,3


,subject,night_id,Model_Eligible,reason,missing_features
1,c1s01,2022-06-12,True,insufficient_presleep_glucose,"mean_glucose_presleep,glucose_slope_presleep"
3,c1s01,2022-06-15,True,insufficient_presleep_hrv,"rmssd_presleep,lfhf_presleep"
6,c1s01,2022-06-26,True,insufficient_presleep_hrv,"rmssd_presleep,lfhf_presleep"
9,c1s02,2022-05-23,True,insufficient_presleep_hrv,"rmssd_presleep,lfhf_presleep"
10,c1s02,2022-05-26,True,insufficient_presleep_hrv,"rmssd_presleep,lfhf_presleep"
21,c1s03,2022-06-04,True,insufficient_presleep_hrv,"rmssd_presleep,lfhf_presleep"
23,c1s03,2022-06-06,True,insufficient_presleep_hrv,"rmssd_presleep,lfhf_presleep"
25,c1s03,2022-06-08,True,insufficient_presleep_hrv,"rmssd_presleep,lfhf_presleep"
32,c1s04,2022-06-12,True,insufficient_presleep_hrv,"rmssd_presleep,lfhf_presleep"
36,c1s05,2022-06-25,True,insufficient_presleep_hrv,"rmssd_presleep,lfhf_presleep"


Glucose values matching the old post-cutoff window: 99
All deliverables: /Users/aryan/Documents/GitHub/SleepGuard/results/lopo/run_20261103T164918_024116Z
Read: /Users/aryan/Documents/GitHub/SleepGuard/results/lopo/run_20261103T164918_024116Z/REPORT.md


## References

- [PhysioCGM public data](https://doi.org/10.6084/m9.figshare.28136294)
- [NeuroKit HRV / Welch LF-HF](https://neuropsychology.github.io/NeuroKit/functions/hrv.html)
- [Average precision definition](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html)
- [Grouped permutation test semantics](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.permutation_test_score.html)

The custom permutation loop is needed to regenerate label-derived histories; shuffling only labels beside fixed history features would not test the actual pipeline.